In [ ]:
# ============================================================
# V2 run/update sweep CSV
# ============================================================

import time
import numpy as np
import pandas as pd

import md_Helpers.Simulation_Helpers as sh
import md_Helpers.Sweep_Helpers as sw
import md_Helpers.Logging_Helpers as lh


# ============================================================
# Sweep settings
# ============================================================

n_fcc_cells = 30

rho_min = 0.50
rho_max = 0.80
rho_step = 0.02

kT_min = 0.70
kT_max = 1.00
kT_step = 0.02

rho_values = sw.make_scan_values(
    rho_min,
    rho_max,
    rho_step,
)

kT_values = sw.make_scan_values(
    kT_min,
    kT_max,
    kT_step,
)

nsteps = 1_000_000
log_period = 1_000
seed = 1
phase_name = "randomization"

overwrite = False
overwrite_lattice = False
skip_completed_with_values = True


# ============================================================
# Summary CSV
# ============================================================

summary_path = sw.get_v2_summary_path(
    n_fcc_cells=n_fcc_cells,
    kT_min=kT_min,
    kT_max=kT_max,
    rho_min=rho_min,
    rho_max=rho_max,
    nsteps=nsteps,
)

df = sw.load_summary_csv(
    summary_path,
)

print("Summary CSV:")
print(summary_path)
print("Existing rows:", len(df))

print("\nrho_values:")
print(rho_values)

print("\nkT_values:")
print(kT_values)


# ============================================================
# Run sweep
# ============================================================

total_runs = len(kT_values) * len(rho_values)
run_counter = 0

sweep_start = time.time()

for kT in kT_values:
    for rho in rho_values:
        run_counter += 1

        kT = float(kT)
        rho = float(rho)

        sweep_key = sw.make_v2_sweep_key(
            n_fcc_cells=n_fcc_cells,
            target_rho=rho,
            kT=kT,
            nsteps=nsteps,
            log_period=log_period,
            seed=seed,
            phase_name=phase_name,
        )

        # ========================================================
        # Expected file paths for this state
        # ========================================================

        expected_paths = lh.get_phase_paths(
            n_fcc_cells=n_fcc_cells,
            target_rho=rho,
            kT=kT,
            nsteps=nsteps,
            seed=seed,
            phase_name=phase_name,
        )

        expected_state_path = expected_paths["state_path"]
        expected_log_path = expected_paths["log_path"]

        state_exists = expected_state_path.exists()
        log_exists = expected_log_path.exists()
        files_exist = state_exists and log_exists

        # ========================================================
        # Print run information
        # ========================================================

        print("\n" + "=" * 80)
        print(f"Run {run_counter} / {total_runs}")
        print(f"n_fcc_cells = {n_fcc_cells}")
        print(f"N           = {4 * n_fcc_cells**3}")
        print(f"kT          = {kT:.2f}")
        print(f"rho         = {rho:.2f}")
        print(f"nsteps      = {nsteps:,}")
        print(f"log_period  = {log_period:,}")
        print(f"state exists: {state_exists}")
        print(f"log exists:   {log_exists}")
        print("=" * 80)

        # ========================================================
        # Skip only if CSV is complete AND files exist
        # ========================================================

        csv_completed = False

        if skip_completed_with_values and not overwrite:
            csv_completed = sw.already_completed_with_values(
                df=df,
                sweep_key=sweep_key,
            )

            if csv_completed and files_exist:
                print("Already completed with values in CSV and files exist. Skipping.")
                continue

            if csv_completed and not files_exist:
                print("CSV says completed, but one or more files are missing.")
                print("Re-running this state and updating the CSV.")
                print("Expected state:", expected_state_path)
                print("Expected log:  ", expected_log_path)

        run_start = time.time()

        row = {
            "sweep_key": sweep_key,
            "n_fcc_cells": n_fcc_cells,
            "N": 4 * n_fcc_cells**3,
            "kT": kT,
            "target_rho": rho,
            "nsteps": nsteps,
            "log_period": log_period,
            "seed": seed,
            "phase_name": phase_name,
            "status": "not_started",
        }

        try:
            result = sh.get_or_make_thermalized_state(
                n_fcc_cells=n_fcc_cells,
                target_rho=rho,
                kT=kT,
                nsteps=nsteps,
                log_period=log_period,
                seed=seed,
                phase_name=phase_name,
                overwrite=overwrite,
                overwrite_lattice=overwrite_lattice,
            )

            run_time_seconds = time.time() - run_start

            row.update(
                sw.summarize_completed_result(
                    result=result,
                    run_time_seconds=run_time_seconds,
                    n_last=100,
                )
            )

            print("Completed")
            print("Created new:", row["created_new"])
            print("Phase separated:", row["phase_separated"])
            print("State file:", row["state_path"])
            print("Log file:  ", row["log_path"])
            print(f"Runtime:   {run_time_seconds:.2f} seconds")

            print("Final last-100 stats:")
            print(f"  pressure mean = {row['pressure_mean_last100']}")
            print(f"  pressure std  = {row['pressure_std_last100']}")
            print(f"  PE/N mean     = {row['PE_per_particle_mean_last100']}")
            print(f"  PE/N std      = {row['PE_per_particle_std_last100']}")
            print(f"  KE/N mean     = {row['KE_per_particle_mean_last100']}")
            print(f"  KE/N std      = {row['KE_per_particle_std_last100']}")

        except Exception as e:
            run_time_seconds = time.time() - run_start

            row.update(
                sw.make_failed_row_update(
                    error=e,
                    run_time_seconds=run_time_seconds,
                )
            )

            print("FAILED")
            print(e)
            print(f"Runtime: {run_time_seconds:.2f} seconds")

        # ========================================================
        # Update CSV after every attempted run
        # ========================================================

        df = sw.update_summary_row(
            df=df,
            row=row,
        )

        df.to_csv(
            summary_path,
            index=False,
        )

        print("Updated summary CSV:")
        print(summary_path)


# ============================================================
# Final summary
# ============================================================

sweep_time_seconds = time.time() - sweep_start

df.to_csv(
    summary_path,
    index=False,
)

print("\n" + "=" * 80)
print("Sweep finished")
print(f"Total grid points:    {total_runs}")
print(f"Rows in CSV:          {len(df)}")
print(f"Completed rows:       {(df['status'] == 'completed').sum()}")
print(f"Failed rows:          {(df['status'] == 'failed').sum()}")
print(f"Total wall time:      {sweep_time_seconds:.2f} seconds")
print("Summary file:")
print(summary_path)
print("=" * 80)

df

In [1]:
# ============================================================
# Adaptive pressure-window density scan
# ============================================================

import time
import traceback
from pathlib import Path
import importlib

import numpy as np
import pandas as pd

import md_Helpers.Project_Paths as pp
import md_Helpers.Create_Lattices as cl
import md_Helpers.Simulation_Helpers as sh
import md_Helpers.Logging_Helpers as lh
import md_Helpers.Sweep_Helpers as swh

importlib.reload(pp)
importlib.reload(cl)
importlib.reload(sh)
importlib.reload(lh)
importlib.reload(swh)


# ============================================================
# Control panel
# ============================================================

n_fcc_cells = 30

kT_start = 0.70
kT_end = 1.00
kT_step = 0.02

nsteps = 1_000_000
log_period = 1_000
seed = 1
phase_name = "randomization"

rho_step = 0.005

# Hard safety bounds so the adaptive scan cannot run forever.
# Change these only if the pressure window lives outside this range.
rho_min_hard = 0.500
rho_max_hard = 0.850

# Desired pressure window
pressure_min = 0.00
pressure_max = 0.15

# Extra buffer so we bracket the window instead of stopping exactly at the edge
pressure_stop_buffer = 0.03
lower_stop = pressure_min - pressure_stop_buffer
upper_stop = pressure_max + pressure_stop_buffer

# Used only if no useful old CSV data exists
initial_rho_guess = 0.700

# Tail statistics
n_last = 100

# Maximum scan distance in each direction from the starting density
max_points_each_direction = 200

# Reuse completed simulations
overwrite = False
overwrite_lattice = False


# Stop scanning downward once the system phase separates.
# This prevents wasting time below the single-phase branch.
stop_down_scan_at_phase_separation = True


# ============================================================
# Small helpers
# ============================================================

def row_is_phase_separated(row):
    """
    Return True if a row is marked as phase separated.
    """

    return swh.clean_bool_value(
        row.get("phase_separated", np.nan)
    ) is True


def snap_density(rho, rho_step=rho_step):
    """
    Snap density to the nearest rho_step grid point.
    """
    return float(np.round(np.round(float(rho) / rho_step) * rho_step, 3))


def clip_density(rho):
    """
    Snap density and keep it inside the hard density bounds.
    """
    rho = snap_density(rho)
    rho = max(rho_min_hard, min(rho_max_hard, rho))
    return snap_density(rho)


def make_adaptive_summary_path():
    """
    Build a clean CSV path for this adaptive sweep.
    """

    summary_folder = (
        Path(pp.THERMALIZED_STATES_V2_ROOT)
        / "FCC"
        / f"n_cells_{int(n_fcc_cells)}"
        / "Adaptive_Pressure_Window_Summaries"
    )

    summary_folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    def fmt(x, ndigits=3):
        return f"{float(x):.{ndigits}f}".replace(".", "p").replace("-", "m")

    summary_path = (
        summary_folder
        / (
            f"adaptive_pressure_window"
            f"_ncells_{int(n_fcc_cells)}"
            f"_kT_{fmt(kT_start, 3)}_{fmt(kT_end, 3)}"
            f"_P_{fmt(pressure_min, 3)}_{fmt(pressure_max, 3)}"
            f"_drho_{fmt(rho_step, 3)}"
            f"_nsteps_{int(nsteps)}"
            f"_seed_{int(seed)}.csv"
        )
    )

    return summary_path


def standardize_summary_columns(df):
    """
    Make older/newer CSV column names easier to use together.
    """

    if df.empty:
        return df

    rename_map = {}

    candidate_map = {
        "Pressure_mean_last100": "pressure_mean_last100",
        "Pressure_std_last100": "pressure_std_last100",

        "PE_mean_last100_per_particle": "PE_per_particle_mean_last100",
        "PE_std_last100_per_particle": "PE_per_particle_std_last100",

        "KE_mean_last100_per_particle": "KE_per_particle_mean_last100",
        "KE_std_last100_per_particle": "KE_per_particle_std_last100",

        "gsd_path": "state_path",
    }

    for old_col, new_col in candidate_map.items():
        if old_col in df.columns and new_col not in df.columns:
            rename_map[old_col] = new_col

    df = df.rename(columns=rename_map)

    for col in [
        "kT",
        "target_rho",
        "actual_rho",
        "pressure_mean_last100",
        "pressure_std_last100",
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce",
            )

    return df


def load_all_previous_summary_data(adaptive_summary_path):
    """
    Load old rectangular sweeps and old adaptive sweeps.
    These are used only to choose better starting densities.
    """

    base = (
        Path(pp.THERMALIZED_STATES_V2_ROOT)
        / "FCC"
        / f"n_cells_{int(n_fcc_cells)}"
    )

    csv_paths = []

    csv_paths += sorted(
        (base / "Sweep_Summaries").glob("*.csv")
    )

    csv_paths += sorted(
        (base / "Adaptive_Pressure_Window_Summaries").glob("*.csv")
    )

    rows = []

    for path in csv_paths:
        try:
            path = Path(path)

            if path.resolve() == Path(adaptive_summary_path).resolve():
                continue

            temp_df = pd.read_csv(path)
            temp_df["source_summary_path"] = str(path)
            rows.append(temp_df)

        except Exception as error:
            print("Could not read previous summary CSV:")
            print(path)
            print(repr(error))

    if len(rows) == 0:
        return pd.DataFrame()

    df = pd.concat(
        rows,
        ignore_index=True,
    )

    df = standardize_summary_columns(df)

    if "status" in df.columns:
        df = df[df["status"] == "completed"].copy()

    if "pressure_mean_last100" in df.columns:
        df = df[np.isfinite(df["pressure_mean_last100"])].copy()

    return df.reset_index(drop=True)


def pressure_region_label(pressure):
    """
    Label where a pressure value sits relative to the target window.
    """

    if not np.isfinite(pressure):
        return "missing_pressure"

    if pressure <= lower_stop:
        return "below_lower_stop"

    if pressure < pressure_min:
        return "below_pressure_window"

    if pressure <= pressure_max:
        return "inside_pressure_window"

    if pressure < upper_stop:
        return "above_pressure_window"

    return "above_upper_stop"


def add_adaptive_flags(row):
    """
    Add adaptive-scan-specific flags to one summary row.
    """

    pressure = row.get(
        "pressure_mean_last100",
        np.nan,
    )

    try:
        pressure = float(pressure)
    except Exception:
        pressure = np.nan

    phase_separated = swh.clean_bool_value(
        row.get("phase_separated", np.nan)
    )

    status = row.get("status", "")

    inside_pressure_window = (
        status == "completed"
        and np.isfinite(pressure)
        and pressure_min <= pressure <= pressure_max
        and phase_separated is not True
    )

    row["pressure_window_min"] = float(pressure_min)
    row["pressure_window_max"] = float(pressure_max)
    row["pressure_lower_stop"] = float(lower_stop)
    row["pressure_upper_stop"] = float(upper_stop)
    row["density_step"] = float(rho_step)

    row["pressure_region"] = pressure_region_label(pressure)
    row["inside_pressure_window"] = bool(inside_pressure_window)

    if "used_for_bracket" not in row:
        row["used_for_bracket"] = False

    return row


def choose_start_density(kT, prior_df, current_df, previous_midpoint=None):
    """
    Choose a starting density.

    Priority:
    1. Previous temperature's successful pressure-window midpoint.
    2. Existing CSV data closest to this kT and closest to middle pressure.
    3. Manual fallback initial_rho_guess.
    """

    pressure_mid = 0.5 * (pressure_min + pressure_max)

    if previous_midpoint is not None and np.isfinite(previous_midpoint):
        return clip_density(previous_midpoint)

    data_frames = []

    if prior_df is not None and len(prior_df) > 0:
        data_frames.append(prior_df.copy())

    if current_df is not None and len(current_df) > 0:
        data_frames.append(current_df.copy())

    if len(data_frames) == 0:
        return clip_density(initial_rho_guess)

    df = pd.concat(
        data_frames,
        ignore_index=True,
    )

    df = standardize_summary_columns(df)

    required = [
        "kT",
        "target_rho",
        "pressure_mean_last100",
    ]

    for col in required:
        if col not in df.columns:
            return clip_density(initial_rho_guess)

    if "status" in df.columns:
        df = df[df["status"] == "completed"].copy()

    df = df[
        np.isfinite(df["kT"])
        & np.isfinite(df["target_rho"])
        & np.isfinite(df["pressure_mean_last100"])
    ].copy()

    if len(df) == 0:
        return clip_density(initial_rho_guess)

    if "phase_separated" in df.columns:
        phase_clean = df["phase_separated"].apply(swh.clean_bool_value)
        non_phase_df = df[phase_clean != True].copy()

        if len(non_phase_df) > 0:
            df = non_phase_df

    exact_df = df[np.isclose(df["kT"], kT, atol=1.0e-9)].copy()

    if len(exact_df) > 0:
        use_df = exact_df
    else:
        closest_kT = df.iloc[
            np.argmin(np.abs(df["kT"].to_numpy() - kT))
        ]["kT"]

        use_df = df[
            np.isclose(df["kT"], closest_kT, atol=1.0e-9)
        ].copy()

    if len(use_df) == 0:
        return clip_density(initial_rho_guess)

    idx = np.argmin(
        np.abs(use_df["pressure_mean_last100"].to_numpy() - pressure_mid)
    )

    rho_guess = use_df.iloc[idx]["target_rho"]

    return clip_density(rho_guess)


def load_or_initialize_adaptive_csv(summary_path):
    """
    Load the adaptive CSV if it exists.
    """

    summary_path = Path(summary_path)

    if summary_path.exists():
        df = pd.read_csv(summary_path)
        df = standardize_summary_columns(df)
        return df

    return pd.DataFrame()


def get_existing_completed_row(df, sweep_key):
    """
    Return a completed useful row from the adaptive CSV if available.
    """

    if df.empty:
        return None

    if "sweep_key" not in df.columns:
        return None

    matches = df[df["sweep_key"] == sweep_key]

    if len(matches) == 0:
        return None

    row = matches.iloc[-1]

    if swh.completed_row_has_values(row):
        return row.to_dict()

    return None


def run_or_load_one_adaptive_point(
    kT,
    target_rho,
    scan_direction,
    adaptive_pass,
    summary_path,
    df,
):
    """
    Run or load one density-temperature point, summarize it,
    and immediately save the adaptive CSV.
    """

    target_rho = clip_density(target_rho)

    sweep_key = swh.make_v2_sweep_key(
        n_fcc_cells=n_fcc_cells,
        target_rho=target_rho,
        kT=kT,
        nsteps=nsteps,
        log_period=log_period,
        seed=seed,
        phase_name=phase_name,
    )

    base_row = {
        "sweep_key": sweep_key,

        "n_fcc_cells": int(n_fcc_cells),
        "N": int(4 * n_fcc_cells**3),

        "target_rho": float(target_rho),
        "kT": float(kT),

        "nsteps": int(nsteps),
        "log_period": int(log_period),
        "seed": int(seed),
        "phase_name": phase_name,

        "lattice_type": "fcc",
        "density_mode": "fixed_N_variable_L",

        "scan_direction": scan_direction,
        "adaptive_pass": adaptive_pass,
    }

    existing_row = get_existing_completed_row(
        df=df,
        sweep_key=sweep_key,
    )

    if existing_row is not None:
        row = existing_row.copy()
        row.update(base_row)
        row = add_adaptive_flags(row)

        df = swh.update_summary_row(
            df=df,
            row=row,
            key_column="sweep_key",
        )

        df.to_csv(
            summary_path,
            index=False,
        )

        pressure = row.get("pressure_mean_last100", np.nan)

        print(
            f"Already completed | "
            f"kT={kT:.3f}, rho={target_rho:.3f}, "
            f"P={pressure:.5f}, "
            f"phase_sep={row.get('phase_separated', np.nan)}"
        )

        return df, row

    start_time = time.time()

    try:
        result = sh.get_or_make_thermalized_state(
            n_fcc_cells=n_fcc_cells,
            target_rho=target_rho,
            kT=kT,
            nsteps=nsteps,
            phase_name=phase_name,
            log_period=log_period,
            seed=seed,
            overwrite=overwrite,
            overwrite_lattice=overwrite_lattice,
        )

        run_time_seconds = time.time() - start_time

        row_update = swh.summarize_completed_result(
            result=result,
            run_time_seconds=run_time_seconds,
            n_last=n_last,
        )

        row = base_row.copy()
        row.update(row_update)

    except Exception as error:
        run_time_seconds = time.time() - start_time

        row_update = swh.make_failed_row_update(
            error=error,
            run_time_seconds=run_time_seconds,
        )

        row = base_row.copy()
        row.update(row_update)

        row["traceback"] = traceback.format_exc()

    row = add_adaptive_flags(row)

    df = swh.update_summary_row(
        df=df,
        row=row,
        key_column="sweep_key",
    )

    df.to_csv(
        summary_path,
        index=False,
    )

    pressure = row.get("pressure_mean_last100", np.nan)

    print(
        f"{row.get('status', 'unknown'):>9} | "
        f"kT={kT:.3f}, rho={target_rho:.3f}, "
        f"P={pressure:.5f}, "
        f"region={row.get('pressure_region', '')}, "
        f"phase_sep={row.get('phase_separated', np.nan)}, "
        f"created_new={row.get('created_new', np.nan)}"
    )

    return df, row


def scan_outward_from_start(
    kT,
    start_rho,
    start_row,
    summary_path,
    df,
):
    """
    Starting from start_rho, scan down and up in alternating steps
    until the pressure window is bracketed.

    New behavior:
    - The downward scan stops if it hits phase separation.
    - This is needed when the low-density side phase separates before
      reaching the lower pressure stop.
    """

    start_pressure = start_row.get(
        "pressure_mean_last100",
        np.nan,
    )

    try:
        start_pressure = float(start_pressure)
    except Exception:
        start_pressure = np.nan

    down_done = False
    up_done = False

    down_stop_reason = ""
    up_stop_reason = ""

    if np.isfinite(start_pressure):
        if start_pressure <= lower_stop:
            down_done = True
            down_stop_reason = "start_below_lower_stop"

        if start_pressure >= upper_stop:
            up_done = True
            up_stop_reason = "start_above_upper_stop"

    if row_is_phase_separated(start_row) and stop_down_scan_at_phase_separation:
        down_done = True
        down_stop_reason = "start_phase_separated"

    if start_rho <= rho_min_hard:
        down_done = True
        down_stop_reason = "start_at_rho_min_hard"

    if start_rho >= rho_max_hard:
        up_done = True
        up_stop_reason = "start_at_rho_max_hard"

    for i in range(1, max_points_each_direction + 1):

        # ========================================================
        # Scan downward in density
        # ========================================================

        if not down_done:
            rho_down = clip_density(start_rho - i * rho_step)

            if rho_down < rho_min_hard or np.isclose(rho_down, rho_min_hard):
                down_done = True
                down_stop_reason = "rho_min_hard"

            df, row_down = run_or_load_one_adaptive_point(
                kT=kT,
                target_rho=rho_down,
                scan_direction="down",
                adaptive_pass="bracket",
                summary_path=summary_path,
                df=df,
            )

            pressure_down = row_down.get(
                "pressure_mean_last100",
                np.nan,
            )

            try:
                pressure_down = float(pressure_down)
            except Exception:
                pressure_down = np.nan

            phase_down = row_is_phase_separated(row_down)

            if phase_down and stop_down_scan_at_phase_separation:
                down_done = True
                down_stop_reason = "phase_separated"

                print(
                    f"Stopping downward scan for kT={kT:.3f}: "
                    f"rho={rho_down:.3f} phase separated before "
                    f"reaching P <= {lower_stop:.3f}."
                )

            elif np.isfinite(pressure_down) and pressure_down <= lower_stop:
                down_done = True
                down_stop_reason = "below_lower_stop"

        # ========================================================
        # Scan upward in density
        # ========================================================

        if not up_done:
            rho_up = clip_density(start_rho + i * rho_step)

            if rho_up > rho_max_hard or np.isclose(rho_up, rho_max_hard):
                up_done = True
                up_stop_reason = "rho_max_hard"

            df, row_up = run_or_load_one_adaptive_point(
                kT=kT,
                target_rho=rho_up,
                scan_direction="up",
                adaptive_pass="bracket",
                summary_path=summary_path,
                df=df,
            )

            pressure_up = row_up.get(
                "pressure_mean_last100",
                np.nan,
            )

            try:
                pressure_up = float(pressure_up)
            except Exception:
                pressure_up = np.nan

            if np.isfinite(pressure_up) and pressure_up >= upper_stop:
                up_done = True
                up_stop_reason = "above_upper_stop"

        if down_done and up_done:
            break

    if not down_done:
        print(
            f"WARNING: kT={kT:.3f} did not finish downward scan. "
            f"It did not reach lower pressure stop, phase separation, "
            f"or density limit before hitting scan limit."
        )
    else:
        print(
            f"Downward scan finished for kT={kT:.3f}: "
            f"{down_stop_reason}"
        )

    if not up_done:
        print(
            f"WARNING: kT={kT:.3f} did not reach upper pressure stop "
            f"P >= {upper_stop:.3f} before hitting scan limit."
        )
    else:
        print(
            f"Upward scan finished for kT={kT:.3f}: "
            f"{up_stop_reason}"
        )

    return df


def get_completed_rows_for_kT(
    df,
    kT,
    require_non_phase_separated=True,
):
    """
    Get completed adaptive rows for one temperature.

    By default, only keep non-phase-separated rows, because the pressure
    window we care about is the usable single-phase branch.
    """

    if df.empty:
        return pd.DataFrame()

    df = standardize_summary_columns(df)

    required = [
        "kT",
        "target_rho",
        "pressure_mean_last100",
        "status",
    ]

    for col in required:
        if col not in df.columns:
            return pd.DataFrame()

    rows = df[
        np.isclose(df["kT"], kT, atol=1.0e-9)
        & (df["status"] == "completed")
    ].copy()

    rows = rows[
        np.isfinite(rows["target_rho"])
        & np.isfinite(rows["pressure_mean_last100"])
    ].copy()

    if require_non_phase_separated and "phase_separated" in rows.columns:
        phase_clean = rows["phase_separated"].apply(
            swh.clean_bool_value
        )

        rows = rows[phase_clean != True].copy()

    return rows.sort_values(
        "target_rho",
        ignore_index=True,
    )


def find_bracket_density_edges(df, kT):
    """
    Find the density interval that brackets the usable pressure window.

    This uses only non-phase-separated completed rows.

    If the low-density side phase separates before reaching lower_stop,
    then we use the lowest available non-phase-separated point inside
    the useful pressure range.
    """

    rows = get_completed_rows_for_kT(
        df=df,
        kT=kT,
        require_non_phase_separated=True,
    )

    if len(rows) == 0:
        return None, None

    below_rows = rows[
        rows["pressure_mean_last100"] <= lower_stop
    ].copy()

    above_rows = rows[
        rows["pressure_mean_last100"] >= upper_stop
    ].copy()

    if len(below_rows) > 0 and len(above_rows) > 0:
        rho_low = below_rows["target_rho"].max()
        rho_high = above_rows["target_rho"].min()

        if rho_low <= rho_high:
            return clip_density(rho_low), clip_density(rho_high)

    useful_rows = rows[
        (rows["pressure_mean_last100"] >= lower_stop)
        & (rows["pressure_mean_last100"] <= upper_stop)
    ].copy()

    if len(useful_rows) > 0:
        rho_low = useful_rows["target_rho"].min()
        rho_high = useful_rows["target_rho"].max()

        print(
            f"Using available non-phase-separated pressure range for "
            f"kT={kT:.3f}: rho={rho_low:.3f} to {rho_high:.3f}. "
            f"This can happen if the low-density side phase separates "
            f"before reaching lower_stop."
        )

        return clip_density(rho_low), clip_density(rho_high)

    return None, None


def density_values_between(rho_low, rho_high):
    """
    Return every density on the rho_step grid between two endpoints.
    """

    i_low = int(round(rho_low / rho_step))
    i_high = int(round(rho_high / rho_step))

    values = [
        snap_density(i * rho_step)
        for i in range(i_low, i_high + 1)
    ]

    values = [
        rho for rho in values
        if rho_min_hard <= rho <= rho_max_hard
    ]

    return values


def fill_missing_density_points_inside_bracket(
    kT,
    summary_path,
    df,
):
    """
    Once the pressure window is bracketed, make sure every 0.005
    density point inside the bracket exists.
    """

    rho_low, rho_high = find_bracket_density_edges(
        df=df,
        kT=kT,
    )

    if rho_low is None or rho_high is None:
        print(
            f"WARNING: Could not identify a complete pressure bracket "
            f"for kT={kT:.3f}. Skipping fill pass."
        )

        return df, None

    print(
        f"\nFill pass for kT={kT:.3f}: "
        f"rho={rho_low:.3f} to rho={rho_high:.3f} "
        f"in steps of {rho_step:.3f}"
    )

    for rho in density_values_between(rho_low, rho_high):

        sweep_key = swh.make_v2_sweep_key(
            n_fcc_cells=n_fcc_cells,
            target_rho=rho,
            kT=kT,
            nsteps=nsteps,
            log_period=log_period,
            seed=seed,
            phase_name=phase_name,
        )

        existing_row = get_existing_completed_row(
            df=df,
            sweep_key=sweep_key,
        )

        if existing_row is not None:
            continue

        df, row = run_or_load_one_adaptive_point(
            kT=kT,
            target_rho=rho,
            scan_direction="fill",
            adaptive_pass="fill_bracket",
            summary_path=summary_path,
            df=df,
        )

    # Mark every completed point inside the final bracket
    if "used_for_bracket" not in df.columns:
        df["used_for_bracket"] = False

    mask = (
        np.isclose(pd.to_numeric(df["kT"], errors="coerce"), kT, atol=1.0e-9)
        & (
            pd.to_numeric(df["target_rho"], errors="coerce")
            >= rho_low - 0.5 * rho_step
        )
        & (
            pd.to_numeric(df["target_rho"], errors="coerce")
            <= rho_high + 0.5 * rho_step
        )
        & (df["status"] == "completed")
    )

    df.loc[mask, "used_for_bracket"] = True

    df.to_csv(
        summary_path,
        index=False,
    )

    bracket_midpoint = 0.5 * (rho_low + rho_high)

    return df, clip_density(bracket_midpoint)


# ============================================================
# Run adaptive scan
# ============================================================

summary_path = make_adaptive_summary_path()

print("Adaptive pressure-window scan")
print("=" * 80)
print("summary_path =", summary_path)
print("n_fcc_cells =", n_fcc_cells)
print("N =", 4 * n_fcc_cells**3)
print("kT range =", kT_start, "to", kT_end, "step", kT_step)
print("rho_step =", rho_step)
print("hard rho bounds =", rho_min_hard, "to", rho_max_hard)
print("pressure window =", pressure_min, "to", pressure_max)
print("pressure stop bounds =", lower_stop, "to", upper_stop)
print("nsteps =", nsteps)
print("log_period =", log_period)
print("seed =", seed)
print("=" * 80)

adaptive_df = load_or_initialize_adaptive_csv(
    summary_path=summary_path,
)

prior_df = load_all_previous_summary_data(
    adaptive_summary_path=summary_path,
)

kT_values = swh.make_scan_values(
    kT_start,
    kT_end,
    kT_step,
    decimals=3,
)

previous_midpoint = None

for kT in kT_values:

    print("\n" + "=" * 80)
    print(f"Starting adaptive scan for kT = {kT:.3f}")
    print("=" * 80)

    start_rho = choose_start_density(
        kT=kT,
        prior_df=prior_df,
        current_df=adaptive_df,
        previous_midpoint=previous_midpoint,
    )

    print(f"Chosen starting density: rho = {start_rho:.3f}")

    adaptive_df, start_row = run_or_load_one_adaptive_point(
        kT=kT,
        target_rho=start_rho,
        scan_direction="start",
        adaptive_pass="bracket",
        summary_path=summary_path,
        df=adaptive_df,
    )

    adaptive_df = scan_outward_from_start(
        kT=kT,
        start_rho=start_rho,
        start_row=start_row,
        summary_path=summary_path,
        df=adaptive_df,
    )

    adaptive_df, previous_midpoint = fill_missing_density_points_inside_bracket(
        kT=kT,
        summary_path=summary_path,
        df=adaptive_df,
    )

    print("\nFinished kT =", f"{kT:.3f}")
    print("Next starting midpoint =", previous_midpoint)


# ============================================================
# Final output
# ============================================================

adaptive_df = pd.read_csv(summary_path)
adaptive_df = standardize_summary_columns(adaptive_df)

print("\n" + "=" * 80)
print("Adaptive scan complete")
print("=" * 80)
print("Saved CSV:")
print(summary_path)
print("=" * 80)

display_columns = [
    "kT",
    "target_rho",
    "actual_rho",
    "pressure_mean_last100",
    "pressure_std_last100",
    "pressure_region",
    "inside_pressure_window",
    "used_for_bracket",
    "phase_separated",
    "status",
    "created_new",
]

display_columns = [
    col for col in display_columns
    if col in adaptive_df.columns
]

display(
    adaptive_df[display_columns]
    .sort_values(["kT", "target_rho"])
    .reset_index(drop=True)
)

Adaptive pressure-window scan
summary_path = /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v2/FCC/n_cells_30/Adaptive_Pressure_Window_Summaries/adaptive_pressure_window_ncells_30_kT_0p700_1p000_P_0p000_0p150_drho_0p005_nsteps_1000000_seed_1.csv
n_fcc_cells = 30
N = 108000
kT range = 0.7 to 1.0 step 0.02
rho_step = 0.005
hard rho bounds = 0.5 to 0.85
pressure window = 0.0 to 0.15
pressure stop bounds = -0.03 to 0.18
nsteps = 1000000
log_period = 1000
seed = 1

Starting adaptive scan for kT = 0.700
Chosen starting density: rho = 0.815
Already completed | kT=0.700, rho=0.815, P=0.05802, phase_sep=False
Already completed | kT=0.700, rho=0.810, P=0.00007, phase_sep=False
Already completed | kT=0.700, rho=0.820, P=0.11956, phase_sep=False
Already completed | kT=0.700, rho=0.805, P=-0.05757, phase_sep=False
Already completed | kT=0.700, rho=0.825, P=0.18582, phase_sep=False
Downward scan finished for kT=0.700: below_lower_stop
Upward scan finished for kT=0.700: above_upper_stop

Fill

,kT,target_rho,actual_rho,pressure_mean_last100,pressure_std_last100,pressure_region,inside_pressure_window,used_for_bracket,phase_separated,status,created_new
0,0.7,0.800,0.800,-0.111740,0.012039,below_lower_stop,False,False,False,completed,False
1,0.7,0.805,0.805,-0.057570,0.010775,below_lower_stop,False,True,False,completed,True
2,0.7,0.810,0.810,0.000068,0.009695,inside_pressure_window,True,True,False,completed,True
3,0.7,0.815,0.815,0.058018,0.011577,inside_pressure_window,True,True,False,completed,True
4,0.7,0.820,0.820,0.119560,0.011564,inside_pressure_window,True,True,False,completed,True
...,...,...,...,...,...,...,...,...,...,...,...
142,1.0,0.650,0.650,0.114540,0.011063,inside_pressure_window,True,True,False,completed,True
143,1.0,0.655,0.655,0.132486,0.011113,inside_pressure_window,True,True,False,completed,True
144,1.0,0.660,0.660,0.152198,0.012085,above_pressure_window,False,True,False,completed,False
145,1.0,0.665,0.665,0.174896,0.010102,above_pressure_window,False,True,False,completed,True
